# Análisis de inscripciones

Este notebook corresponde a la Parte 2 de la prueba técnica.

El objetivo es conectarse a la base de datos MariaDB, cargar las tablas principales en DataFrames de Pandas y realizar análisis sobre alumnos, programas, inscripciones e historial de cambios de estatus.

## 1. Importación de librerías

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from getpass import getpass

## 2. Configuración de conexión a MariaDB

La conexión se configura con variables simples para que pueda adaptarse fácilmente a otro entorno local.
La contraseña se solicita en tiempo de ejecución para evitar dejarla escrita dentro del notebook.

In [5]:
DB_USER = "python_user"
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "inscripciones_db"

DB_PASSWORD = getpass("Contraseña de MariaDB: ")

In [7]:
connection_url = URL.create(
    drivername="mysql+pymysql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

engine = create_engine(connection_url)

## 3. Carga de tablas en DataFrames

In [8]:
df_alumnos = pd.read_sql("SELECT * FROM alumnos", engine)
df_programas = pd.read_sql("SELECT * FROM programas", engine)
df_inscripciones = pd.read_sql("SELECT * FROM inscripciones", engine)
df_historial = pd.read_sql("SELECT * FROM historial_estatus", engine)

## 4. Verificación inicial de datos cargados

In [9]:
print("Alumnos:", df_alumnos.shape)
print("Programas:", df_programas.shape)
print("Inscripciones:", df_inscripciones.shape)
print("Historial:", df_historial.shape)

Alumnos: (465, 4)
Programas: (10, 2)
Inscripciones: (465, 5)
Historial: (15, 6)


In [10]:
display(df_alumnos.head())
display(df_programas.head())
display(df_inscripciones.head())
display(df_historial.head())

,id_alumno,nombre,empresa,fecha_ingreso
0,1001,Luis Cruz,Soriana,2025-02-26
1,1002,Andrés Ramírez,Coppel,2022-10-04
2,1003,Daniela Ortiz,Soriana,2017-10-02
3,1004,Luis Reyes,Soriana,2015-06-06
4,1005,Sofía Hernández,Soriana,2017-03-31


,id_programa,nombre_programa
0,8,Bachillerato Ejecutivo
1,10,Bachillerato General
2,5,Lic. Administración
3,2,Lic. Contaduría
4,3,Lic. Logística


,id_inscripcion,id_alumno,id_programa,estatus_actual,fecha_inscripcion
0,1,1001,1,suspendido,2025-02-26
1,2,1002,2,suspendido,2022-10-04
2,3,1003,3,egresado,2017-10-02
3,4,1004,3,egresado,2015-06-06
4,5,1005,3,baja_programa,2017-03-31


,id_historial,id_inscripcion,estatus_anterior,estatus_nuevo,fecha_cambio,motivo
0,1,1,inscrito,activo,2026-02-28,Inicio regular del programa
1,2,1,activo,baja_empresa,2026-06-03,Cambio laboral reportado por la empresa
2,3,1,baja_empresa,activo,2026-06-23,Reingreso autorizado por reincorporación laboral
3,4,7,inscrito,activo,2026-03-30,Activación inicial de inscripción
4,5,7,activo,baja_empresa,2026-06-16,Finalización de relación laboral con la empresa


## 5. Construcción del DataFrame consolidado

Se construye un DataFrame principal combinando alumnos, inscripciones y programas.

Este DataFrame será la base para los análisis de distribución de estatus, tasas por programa y evolución de inscripciones.

In [11]:
df_base = (
    df_inscripciones
    .merge(df_alumnos, on="id_alumno", how="inner")
    .merge(df_programas, on="id_programa", how="inner")
)

df_base.head()

,id_inscripcion,id_alumno,id_programa,estatus_actual,fecha_inscripcion,nombre,empresa,fecha_ingreso,nombre_programa
0,1,1001,1,suspendido,2025-02-26,Luis Cruz,Soriana,2025-02-26,Lic. Negocios
1,2,1002,2,suspendido,2022-10-04,Andrés Ramírez,Coppel,2022-10-04,Lic. Contaduría
2,3,1003,3,egresado,2017-10-02,Daniela Ortiz,Soriana,2017-10-02,Lic. Logística
3,4,1004,3,egresado,2015-06-06,Luis Reyes,Soriana,2015-06-06,Lic. Logística
4,5,1005,3,baja_programa,2017-03-31,Sofía Hernández,Soriana,2017-03-31,Lic. Logística


## 6. Revisión de estructura del DataFrame base

In [12]:
print("Filas y columnas:", df_base.shape)

df_base.info()

Filas y columnas: (465, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 465 entries, 0 to 464
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_inscripcion     465 non-null    int64 
 1   id_alumno          465 non-null    int64 
 2   id_programa        465 non-null    int64 
 3   estatus_actual     465 non-null    object
 4   fecha_inscripcion  465 non-null    object
 5   nombre             465 non-null    object
 6   empresa            465 non-null    object
 7   fecha_ingreso      465 non-null    object
 8   nombre_programa    465 non-null    object
dtypes: int64(3), object(6)
memory usage: 32.8+ KB


In [13]:
df_base[[
    "id_inscripcion",
    "id_alumno",
    "nombre",
    "empresa",
    "nombre_programa",
    "estatus_actual",
    "fecha_ingreso",
    "fecha_inscripcion"
]].head(10)

,id_inscripcion,id_alumno,nombre,empresa,nombre_programa,estatus_actual,fecha_ingreso,fecha_inscripcion
0,1,1001,Luis Cruz,Soriana,Lic. Negocios,suspendido,2025-02-26,2025-02-26
1,2,1002,Andrés Ramírez,Coppel,Lic. Contaduría,suspendido,2022-10-04,2022-10-04
2,3,1003,Daniela Ortiz,Soriana,Lic. Logística,egresado,2017-10-02,2017-10-02
3,4,1004,Luis Reyes,Soriana,Lic. Logística,egresado,2015-06-06,2015-06-06
4,5,1005,Sofía Hernández,Soriana,Lic. Logística,baja_programa,2017-03-31,2017-03-31
5,6,1006,Paola Mendoza,Bayer,Secundaria Abierta,baja_programa,2023-03-28,2023-03-28
6,7,1007,Gabriela Cruz,Soriana,Lic. Negocios,baja_empresa,2024-03-30,2024-03-30
7,8,1008,Gabriela Morales,Soriana,Lic. Negocios,baja_empresa,2018-07-05,2018-07-05
8,9,1009,Paola López,Soriana,Lic. Logística,egresado,2017-06-01,2017-06-01
9,10,1010,Valeria Torres,Soriana,Lic. Logística,egresado,2020-11-04,2020-11-04


## 7. Validación de consistencia

In [14]:
print("Inscripciones originales:", df_inscripciones.shape[0])
print("Inscripciones en df_base:", df_base.shape[0])

if df_inscripciones.shape[0] == df_base.shape[0]:
    print("Validación correcta: no se perdieron ni duplicaron inscripciones.")
else:
    print("Revisar joins: la cantidad de inscripciones no coincide.")

Inscripciones originales: 465
Inscripciones en df_base: 465
Validación correcta: no se perdieron ni duplicaron inscripciones.


## 8. Conversión de fechas

In [15]:
df_base["fecha_ingreso"] = pd.to_datetime(df_base["fecha_ingreso"])
df_base["fecha_inscripcion"] = pd.to_datetime(df_base["fecha_inscripcion"])

df_historial["fecha_cambio"] = pd.to_datetime(df_historial["fecha_cambio"])

print("Tipos de datos en df_base:")
display(df_base.dtypes)

print("Tipos de datos en df_historial:")
display(df_historial.dtypes)

Tipos de datos en df_base:


id_inscripcion                int64
id_alumno                     int64
id_programa                   int64
estatus_actual               object
fecha_inscripcion    datetime64[ns]
nombre                       object
empresa                      object
fecha_ingreso        datetime64[ns]
nombre_programa              object
dtype: object

Tipos de datos en df_historial:


id_historial                 int64
id_inscripcion               int64
estatus_anterior            object
estatus_nuevo               object
fecha_cambio        datetime64[ns]
motivo                      object
dtype: object

## 9. Distribución de estatus actual por programa

Se analiza cuántos alumnos hay en cada estatus actual dentro de cada programa académico.

Este análisis permite identificar la composición actual de los programas y detectar rápidamente programas con mayor cantidad de alumnos activos, suspendidos, egresados o dados de baja.

In [16]:
distribucion_estatus = (
    df_base
    .groupby(["nombre_programa", "estatus_actual"])
    .size()
    .reset_index(name="cantidad_alumnos")
    .sort_values(["nombre_programa", "cantidad_alumnos"], ascending=[True, False])
)

distribucion_estatus

,nombre_programa,estatus_actual,cantidad_alumnos
3,Bachillerato Ejecutivo,egresado,11
0,Bachillerato Ejecutivo,activo,2
1,Bachillerato Ejecutivo,baja_empresa,2
2,Bachillerato Ejecutivo,baja_programa,2
4,Bachillerato Ejecutivo,reingreso,1
5,Bachillerato General,baja_empresa,13
6,Bachillerato General,baja_programa,2
7,Bachillerato General,egresado,2
8,Lic. Administración,activo,22
10,Lic. Administración,baja_programa,11


In [ ]:
# Tabla pivot para visualizar la distribución de estatus por programa
# y reutilizarla luego en la gráfica de barras apiladas.

pivot_estatus_programa = pd.pivot_table(
    df_base,
    index="nombre_programa",
    columns="estatus_actual",
    values="id_inscripcion",
    aggfunc="count",
    fill_value=0
)

pivot_estatus_programa

estatus_actual,activo,baja_empresa,baja_programa,egresado,inscrito,reingreso,suspendido
nombre_programa,,,,,,,
Bachillerato Ejecutivo,2,2,2,11,0,1,0
Bachillerato General,0,13,2,2,0,0,0
Lic. Administración,22,2,11,1,0,1,1
Lic. Contaduría,3,2,0,0,2,2,5
Lic. Logística,11,24,84,46,4,6,4
Lic. Mercadotecnia,1,0,8,3,0,0,4
Lic. Negocios,18,33,50,7,7,4,9
Maestría en Dirección,1,5,11,3,0,0,2
Maestría en Educación,4,1,0,5,0,1,7


In [ ]:
# Verificación: total de inscripciones por programa según la tabla pivot.

total_por_programa = pivot_estatus_programa.sum(axis=1).sort_values(ascending=False)

total_por_programa

nombre_programa
Lic. Logística            179
Lic. Negocios             128
Lic. Administración        38
Maestría en Dirección      22
Bachillerato Ejecutivo     18
Maestría en Educación      18
Bachillerato General       17
Lic. Mercadotecnia         16
Secundaria Abierta         15
Lic. Contaduría            14
dtype: int64

## 10. Evolución mensual de altas y bajas

Se analiza la evolución mensual de inscripciones y bajas.

Las altas se calculan a partir de la fecha de inscripción de cada alumno. Las bajas se calculan a partir del historial de cambios de estatus, considerando como bajas los movimientos hacia `baja_empresa` o `baja_programa`.

Este análisis permite observar la relación entre nuevos ingresos y salidas del programa a lo largo del tiempo.

In [19]:
# Altas o inscripciones por mes

altas_por_mes = (
    df_base
    .assign(mes=df_base["fecha_inscripcion"].dt.to_period("M").dt.to_timestamp())
    .groupby("mes")
    .size()
    .reset_index(name="altas")
)

altas_por_mes.head()

,mes,altas
0,2013-08-01,2
1,2013-12-01,1
2,2014-02-01,3
3,2014-03-01,1
4,2014-04-01,1


In [20]:
# Bajas por mes según movimientos del historial

bajas_por_mes = (
    df_historial[df_historial["estatus_nuevo"].isin(["baja_empresa", "baja_programa"])]
    .assign(mes=lambda df: df["fecha_cambio"].dt.to_period("M").dt.to_timestamp())
    .groupby("mes")
    .size()
    .reset_index(name="bajas")
)

bajas_por_mes.head()

,mes,bajas
0,2026-04-01,1
1,2026-06-01,3


In [21]:
# Consolidación mensual de altas y bajas

evolucion_mensual = (
    altas_por_mes
    .merge(bajas_por_mes, on="mes", how="outer")
    .fillna(0)
    .sort_values("mes")
)

evolucion_mensual["altas"] = evolucion_mensual["altas"].astype(int)
evolucion_mensual["bajas"] = evolucion_mensual["bajas"].astype(int)

evolucion_mensual

,mes,altas,bajas
0,2013-08-01,2,0
1,2013-12-01,1,0
2,2014-02-01,3,0
3,2014-03-01,1,0
4,2014-04-01,1,0
...,...,...,...
132,2025-12-01,2,0
133,2026-02-01,3,0
134,2026-04-01,4,1
135,2026-05-01,3,0


In [22]:
print("Total de altas registradas:", evolucion_mensual["altas"].sum())
print("Total de bajas registradas en historial:", evolucion_mensual["bajas"].sum())

Total de altas registradas: 465
Total de bajas registradas en historial: 4


## 11. Tasa de activos por programa

Se calcula la proporción de alumnos activos sobre el total de inscripciones históricas de cada programa.

La fórmula utilizada es:

tasa_activos = alumnos activos / total de inscripciones del programa * 100

Este análisis permite identificar qué programas mantienen una mayor proporción de alumnos activos en relación con su volumen total de inscripciones.

In [23]:
tasa_activos_programa = (
    df_base
    .groupby("nombre_programa")
    .agg(
        total_inscripciones=("id_inscripcion", "count"),
        alumnos_activos=("estatus_actual", lambda x: (x == "activo").sum())
    )
    .reset_index()
)

tasa_activos_programa["tasa_activos_porcentaje"] = (
    tasa_activos_programa["alumnos_activos"]
    / tasa_activos_programa["total_inscripciones"]
    * 100
).round(2)

tasa_activos_programa = tasa_activos_programa.sort_values(
    "tasa_activos_porcentaje",
    ascending=False
)

tasa_activos_programa

,nombre_programa,total_inscripciones,alumnos_activos,tasa_activos_porcentaje
2,Lic. Administración,38,22,57.89
8,Maestría en Educación,18,4,22.22
3,Lic. Contaduría,14,3,21.43
6,Lic. Negocios,128,18,14.06
0,Bachillerato Ejecutivo,18,2,11.11
9,Secundaria Abierta,15,1,6.67
5,Lic. Mercadotecnia,16,1,6.25
4,Lic. Logística,179,11,6.15
7,Maestría en Dirección,22,1,4.55
1,Bachillerato General,17,0,0.00


In [24]:
print("Total de programas analizados:", tasa_activos_programa.shape[0])
print("Tasa máxima:", tasa_activos_programa["tasa_activos_porcentaje"].max())
print("Tasa mínima:", tasa_activos_programa["tasa_activos_porcentaje"].min())

Total de programas analizados: 10
Tasa máxima: 57.89
Tasa mínima: 0.0


## 12. Motivo de baja más frecuente

Se analiza la tabla de historial de estatus para identificar cuál es el motivo de baja más frecuente.

Para este análisis se consideran únicamente los movimientos cuyo estatus nuevo corresponde a una baja:

- baja_empresa
- baja_programa

Este análisis permite detectar patrones recurrentes asociados a la salida de alumnos del programa.

In [25]:
df_bajas = df_historial[
    df_historial["estatus_nuevo"].isin(["baja_empresa", "baja_programa"])
].copy()

df_bajas

,id_historial,id_inscripcion,estatus_anterior,estatus_nuevo,fecha_cambio,motivo
1,2,1,activo,baja_empresa,2026-06-03,Cambio laboral reportado por la empresa
4,5,7,activo,baja_empresa,2026-06-16,Finalización de relación laboral con la empresa
6,7,5,activo,baja_programa,2026-06-08,Baja voluntaria solicitada por el alumno
12,13,24,activo,baja_empresa,2026-04-29,Pausa por cambio de condiciones laborales


In [26]:
motivos_baja = (
    df_bajas
    .groupby("motivo")
    .size()
    .reset_index(name="cantidad")
    .sort_values("cantidad", ascending=False)
)

motivos_baja

,motivo,cantidad
0,Baja voluntaria solicitada por el alumno,1
1,Cambio laboral reportado por la empresa,1
2,Finalización de relación laboral con la empresa,1
3,Pausa por cambio de condiciones laborales,1


In [28]:
if not motivos_baja.empty:
    max_cantidad = motivos_baja["cantidad"].max()

    motivos_mas_frecuentes = motivos_baja[
        motivos_baja["cantidad"] == max_cantidad
    ]

    print("Motivo(s) de baja más frecuente(s):")
    display(motivos_mas_frecuentes)

    if motivos_mas_frecuentes.shape[0] > 1:
        print(f"Hay un empate entre {motivos_mas_frecuentes.shape[0]} motivos, todos con {max_cantidad} ocurrencia(s).")
    else:
        print("Motivo:", motivos_mas_frecuentes.iloc[0]["motivo"])
        print("Cantidad:", max_cantidad)
else:
    print("No se encontraron movimientos de baja en el historial.")

Motivo(s) de baja más frecuente(s):


,motivo,cantidad
0,Baja voluntaria solicitada por el alumno,1
1,Cambio laboral reportado por la empresa,1
2,Finalización de relación laboral con la empresa,1
3,Pausa por cambio de condiciones laborales,1


Hay un empate entre 4 motivos, todos con 1 ocurrencia(s).
